# Task 113: diffpy_pdf — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Pair distribution function refinement for atomic structure

**Paper**: Complex modeling: a strategy and software program for combining multiple information sources to solve ill posed structure and nanostructure inverse problems
**Repository**: https://github.com/diffpy/diffpy.srfit

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 54.09 dB
- **SSIM**: N/A
- **Evaluation**: Pair Distribution Function — least-squares fitting of FCC Cu structural parameters from G(r) (PSNR + CC + parameter errors)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
diffpy_pdf — Pair Distribution Function Analysis
==================================================
From powder diffraction data, extract the atomic pair distribution function G(r)
and fit structural parameters (lattice constant, atomic displacement, coordination).

Physics:
  - Forward: PDF from structure
    G(r) = Σ_n A_n × Gaussian(r - d_n, σ_n)
    Peak positions correspond to interatomic distances in the crystal.
    For FCC: d_n = a × {1/√2, 1, √(3/2), √2, √(5/2), √3, ...}
  - Inverse: Least-squares fitting of structural parameters
    (lattice constant a, Debye-Waller factor B, scale)
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`compute_pdf`, `residual_func`, `main`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 2. FORWARD MODEL: Compute PDF G(r)
# ═══════════════════════════════════════════════════════════════════
def compute_pdf(r, a, B, scale, r_max=30.0):
    """
    Compute the reduced pair distribution function G(r) for FCC.

    G(r) = (1/r) Σ_n [N_n / (4πr_n²ρ₀)] × Gaussian(r - r_n, σ_n) - 4πrρ₀

    Simplified as sum of Gaussian peaks:
    G(r) = scale × Σ_n C_n × exp(-(r - d_n)² / (2σ_n²)) / (σ_n√(2π) × r)

    where σ_n² = B (isotropic Debye-Waller), C_n = coordination number
    """
    shells = fcc_neighbor_distances(a, r_max)
    sigma = np.sqrt(B)  # thermal broadening

    G = np.zeros_like(r)
    rho0 = 4 / a**3  # FCC number density

    for d_n, coord_n in shells:
        # Peak amplitude: proportional to coordination / (4πr²ρ₀)
        # Broadened by Gaussian with width sigma that increases with distance
        sigma_n = sigma * np.sqrt(1 + 0.002 * d_n**2)  # slight distance dependence
        amplitude = coord_n / (4 * np.pi * d_n**2 * rho0)
        peak = amplitude * np.exp(-0.5 * ((r - d_n) / sigma_n)**2) / (sigma_n * np.sqrt(2 * np.pi))
        G += peak

    # Subtract baseline (4πrρ₀ term)
    G = scale * G / np.max(np.abs(G) + 1e-12) - 0  # normalize
    # Apply envelope damping for finite Q range
    G *= np.exp(-0.01 * r**2)

    return G

# ═══════════════════════════════════════════════════════════════════
# 3. INVERSE: Least-squares fitting
# ═══════════════════════════════════════════════════════════════════
def residual_func(params, r, G_measured):
    """Residual for least-squares fitting."""
    a, B, scale = params
    if a < 2.0 or a > 6.0 or B < 0.01 or B > 2.0 or scale < 0.1 or scale > 5.0:
        return np.ones_like(r) * 1e6
    G_model = compute_pdf(r, a, B, scale, r_max=R_MAX)
    return G_model - G_measured

# ═══════════════════════════════════════════════════════════════════
# 6. MAIN
# ═══════════════════════════════════════════════════════════════════
def main():
    print("=" * 60)
    print("diffpy_pdf — Pair Distribution Function Analysis")
    print("=" * 60)

    # 1. r grid
    r = np.arange(R_MIN, R_MAX, DR)

    # 2. Ground truth G(r)
    print("[1/4] Computing ground truth G(r) for FCC Cu ...")
    G_gt = compute_pdf(r, A_TRUE, B_TRUE, SCALE_TRUE, R_MAX)

    # 3. Add noise
    print("[2/4] Adding measurement noise ...")
    noise = NOISE_LEVEL * np.max(np.abs(G_gt)) * np.random.randn(len(r))
    G_noisy = G_gt + noise

    # 4. Fit
    print("[3/4] Fitting structural parameters ...")
    a_fit, B_fit, scale_fit, G_fit, result = fit_pdf(r, G_noisy)
    print(f"  Fitted a     = {a_fit:.4f} Å  (true: {A_TRUE:.4f})")
    print(f"  Fitted B     = {B_fit:.4f} Å² (true: {B_TRUE:.4f})")
    print(f"  Fitted scale = {scale_fit:.4f}   (true: {SCALE_TRUE:.4f})")

    # 5. Metrics
    metrics = compute_metrics(r, G_gt, G_fit, A_TRUE, a_fit, B_TRUE, B_fit, SCALE_TRUE, scale_fit)
    print(f"  PSNR = {metrics['PSNR']:.2f} dB")
    print(f"  CC   = {metrics['CC']:.4f}")

    # 6. Save
    print("[4/4] Saving results ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        np.save(os.path.join(d, "gt_output.npy"), G_gt)
        np.save(os.path.join(d, "recon_output.npy"), G_fit)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    visualize(r, G_gt, G_noisy, G_fit, metrics)

    print("Done ✓")
    return metrics

## 4. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `fcc_neighbor_distances`, `fit_pdf`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 1. FCC neighbor distances
# ═══════════════════════════════════════════════════════════════════
def fcc_neighbor_distances(a, r_max, max_shell=200):
    """
    Compute interatomic distances and coordination numbers for FCC structure.

    For FCC, neighbor distances are a × sqrt(n/2) for certain n values.
    Returns list of (distance, coordination_number) pairs.
    """
    distances = []
    # Generate all lattice vectors and compute distances
    n_max = int(np.ceil(r_max / a)) + 1
    for h in range(-n_max, n_max + 1):
        for k in range(-n_max, n_max + 1):
            for l in range(-n_max, n_max + 1):
                # FCC basis positions: (0,0,0), (0.5,0.5,0), (0.5,0,0.5), (0,0.5,0.5)
                for bx, by, bz in [(0, 0, 0), (0.5, 0.5, 0), (0.5, 0, 0.5), (0, 0.5, 0.5)]:
                    x = (h + bx) * a
                    y = (k + by) * a
                    z = (l + bz) * a
                    d = np.sqrt(x**2 + y**2 + z**2)
                    if 0.1 < d < r_max:
                        distances.append(d)

    # Bin into shells
    distances = np.sort(distances)
    shells = []
    tol = 0.01
    i = 0
    while i < len(distances) and len(shells) < max_shell:
        d_ref = distances[i]
        count = 0
        while i < len(distances) and abs(distances[i] - d_ref) < tol:
            count += 1
            i += 1
        shells.append((d_ref, count))

    return shells

def fit_pdf(r, G_measured):
    """
    Fit structural parameters from measured G(r) using least-squares.

    Parameters to fit:
      - a: lattice constant
      - B: Debye-Waller factor
      - scale: overall scale factor
    """
    # Initial guess (slightly off from true values)
    a0 = 3.5
    B0 = 0.4
    scale0 = 0.8

    result = least_squares(
        residual_func,
        x0=[a0, B0, scale0],
        args=(r, G_measured),
        bounds=([2.0, 0.01, 0.1], [6.0, 2.0, 5.0]),
        method="trf",
        max_nfev=2000,
    )

    a_fit, B_fit, scale_fit = result.x
    G_fitted = compute_pdf(r, a_fit, B_fit, scale_fit, r_max=R_MAX)

    return a_fit, B_fit, scale_fit, G_fitted, result

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 4. METRICS
# ═══════════════════════════════════════════════════════════════════
def compute_metrics(r, G_gt, G_fit, a_true, a_fit, B_true, B_fit, scale_true, scale_fit):
    """Compute PSNR, CC, parameter errors."""
    # Normalize
    g_max = np.max(np.abs(G_gt)) + 1e-12
    gt_n = G_gt / g_max
    fi_n = G_fit / g_max

    # PSNR
    mse = np.mean((gt_n - fi_n)**2)
    psnr = 10 * np.log10(1.0 / (mse + 1e-12))

    # CC
    g = gt_n - np.mean(gt_n)
    f = fi_n - np.mean(fi_n)
    cc = np.sum(g * f) / (np.sqrt(np.sum(g**2) * np.sum(f**2)) + 1e-12)

    # Parameter relative errors
    a_err = abs(a_fit - a_true) / a_true * 100
    B_err = abs(B_fit - B_true) / B_true * 100
    scale_err = abs(scale_fit - scale_true) / scale_true * 100

    return {
        "PSNR": float(psnr),
        "CC": float(cc),
        "lattice_constant_true": float(a_true),
        "lattice_constant_fitted": float(a_fit),
        "lattice_constant_error_pct": float(a_err),
        "debye_waller_true": float(B_true),
        "debye_waller_fitted": float(B_fit),
        "debye_waller_error_pct": float(B_err),
        "scale_true": float(scale_true),
        "scale_fitted": float(scale_fit),
        "scale_error_pct": float(scale_err),
        "RMSE": float(np.sqrt(mse)),
    }

# ═══════════════════════════════════════════════════════════════════
# 5. VISUALIZATION
# ═══════════════════════════════════════════════════════════════════
def visualize(r, G_gt, G_noisy, G_fit, metrics):
    """Plot GT vs fitted G(r) with peak identification."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Panel 1: GT and fitted G(r)
    axes[0, 0].plot(r, G_gt, "b-", linewidth=1.5, label="Ground Truth G(r)")
    axes[0, 0].plot(r, G_fit, "r--", linewidth=1.5, label="Fitted G(r)")
    axes[0, 0].set_xlabel("r (Å)", fontsize=12)
    axes[0, 0].set_ylabel("G(r)", fontsize=12)
    axes[0, 0].set_title("PDF: Ground Truth vs Fit", fontsize=14)
    axes[0, 0].legend(fontsize=11)
    axes[0, 0].set_xlim([R_MIN, 15])

    # Panel 2: Noisy data vs fit
    axes[0, 1].plot(r, G_noisy, "gray", alpha=0.6, linewidth=0.8, label="Noisy data")
    axes[0, 1].plot(r, G_fit, "r-", linewidth=1.5, label="Fitted G(r)")
    axes[0, 1].set_xlabel("r (Å)", fontsize=12)
    axes[0, 1].set_ylabel("G(r)", fontsize=12)
    axes[0, 1].set_title("Noisy Data vs Fit", fontsize=14)
    axes[0, 1].legend(fontsize=11)
    axes[0, 1].set_xlim([R_MIN, 15])

    # Panel 3: Residual
    residual = G_gt - G_fit
    axes[1, 0].plot(r, residual, "g-", linewidth=1.0)
    axes[1, 0].axhline(y=0, color="k", linestyle="--", alpha=0.5)
    axes[1, 0].set_xlabel("r (Å)", fontsize=12)
    axes[1, 0].set_ylabel("Residual", fontsize=12)
    axes[1, 0].set_title(f"Residual (RMSE = {metrics['RMSE']:.4f})", fontsize=14)
    axes[1, 0].set_xlim([R_MIN, 15])

    # Panel 4: Parameter table
    axes[1, 1].axis("off")
    table_data = [
        ["Parameter", "True", "Fitted", "Error (%)"],
        ["a (Å)", f"{metrics['lattice_constant_true']:.4f}",
         f"{metrics['lattice_constant_fitted']:.4f}",
         f"{metrics['lattice_constant_error_pct']:.2f}%"],
        ["B (Å²)", f"{metrics['debye_waller_true']:.4f}",
         f"{metrics['debye_waller_fitted']:.4f}",
         f"{metrics['debye_waller_error_pct']:.2f}%"],
        ["Scale", f"{metrics['scale_true']:.4f}",
         f"{metrics['scale_fitted']:.4f}",
         f"{metrics['scale_error_pct']:.2f}%"],
        ["", "", "", ""],
        ["PSNR", f"{metrics['PSNR']:.2f} dB", "", ""],
        ["CC", f"{metrics['CC']:.4f}", "", ""],
    ]
    table = axes[1, 1].table(
        cellText=table_data, loc="center", cellLoc="center",
        colWidths=[0.25, 0.25, 0.25, 0.25]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.5)
    axes[1, 1].set_title("Fitted Parameters", fontsize=14)

    plt.tight_layout()
    for p in [os.path.join(RESULTS_DIR, "reconstruction_result.png"),
              os.path.join(ASSETS_DIR, "reconstruction_result.png"),
              os.path.join(ASSETS_DIR, "vis_result.png")]:
        plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("diffpy_pdf — Pair Distribution Function Analysis")
print("=" * 60)

### 1. r grid

Intermediate processing step in the pipeline.

In [ ]:
# 1. r grid
r = np.arange(R_MIN, R_MAX, DR)

### 2. Ground truth G(r)

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 2. Ground truth G(r)
print("[1/4] Computing ground truth G(r) for FCC Cu ...")
G_gt = compute_pdf(r, A_TRUE, B_TRUE, SCALE_TRUE, R_MAX)

### 3. Add noise

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 3. Add noise
print("[2/4] Adding measurement noise ...")
noise = NOISE_LEVEL * np.max(np.abs(G_gt)) * np.random.randn(len(r))
G_noisy = G_gt + noise

### 4. Fit

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# 4. Fit
print("[3/4] Fitting structural parameters ...")
a_fit, B_fit, scale_fit, G_fit, result = fit_pdf(r, G_noisy)
print(f"  Fitted a     = {a_fit:.4f} Å  (true: {A_TRUE:.4f})")
print(f"  Fitted B     = {B_fit:.4f} Å² (true: {B_TRUE:.4f})")
print(f"  Fitted scale = {scale_fit:.4f}   (true: {SCALE_TRUE:.4f})")

### 5. Metrics

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# 5. Metrics
metrics = compute_metrics(r, G_gt, G_fit, A_TRUE, a_fit, B_TRUE, B_fit, SCALE_TRUE, scale_fit)
print(f"  PSNR = {metrics['PSNR']:.2f} dB")
print(f"  CC   = {metrics['CC']:.4f}")

### 6. Save

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 6. Save
print("[4/4] Saving results ...")
for d in [RESULTS_DIR, ASSETS_DIR]:
    np.save(os.path.join(d, "gt_output.npy"), G_gt)
    np.save(os.path.join(d, "recon_output.npy"), G_fit)
    with open(os.path.join(d, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

visualize(r, G_gt, G_noisy, G_fit, metrics)

print("Done ✓")
return metrics

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **diffpy_pdf**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=54.09 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Complex modeling: a strategy and software program for combining multiple information sources to solve ill posed structure and nanostructure inverse problems
- Repository: https://github.com/diffpy/diffpy.srfit
